# Station-MAE — Pipeline Exploration
Step-by-step walkthrough of the data, embeddings and encoder.
Run cells top to bottom after setting `ROOT` and `PATH_SWISSSHAPE`.

In [1]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from torch.utils.data import DataLoader

# Project modules
from data.dataset import (
    load_peakweather,
    StationMAEDataset,
    build_spatial_features,
    build_observations,
    VARIABLE_NAMES,
    NUM_VARIABLES,
)
from data.visualize import plot_stations_on_dem, markers_from_stations_table
from model.embeddings import (
    encode_temporal,
    SpatialEmbedding,
    TemporalEmbedding,
    VariableProjection,
)
from model.encoder import StationMAEEncoder

print('Imports OK')

Imports OK


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
ROOT            = "/Users/aureliedejong/Documents/ETH/_DAS Project/PeakWeatherDataset"
PATH_SWISSSHAPE = "/Users/aureliedejong/Documents/ETH/_DAS Project/swissboundaries3d_2026-01_2056_5728.shp.zip"

WINDOW_SIZE  = 12   # 2 hours of input context
DELTA_STEPS  = 6    # predict 1 hour ahead
BATCH_SIZE   = 4
D_MODEL      = 128

## 1 — Load PeakWeather dataset

In [ ]:
ds_peak = load_peakweather(ROOT)
print(ds_peak)
print(f"\nStations : {ds_peak.num_stations}")
print(f"Timesteps: {ds_peak.num_time_steps}")
print(f"\nParameters available:")
ds_peak.show_parameters_description()

In [ ]:
# Stations table overview
ds_peak.stations_table.head()

## 2 — Visualise stations on DEM

In [ ]:
all_markers = markers_from_stations_table(
    ds_peak.stations_table,
    default_color="steelblue",
    size=20,
    label="Meteo stations",
)

plot_stations_on_dem(
    ds_peak,
    stations=all_markers,
    path_swissshape=PATH_SWISSSHAPE,
    title="PeakWeather — All Meteo Stations",
)
plt.show()

## 3 — Spatial features

In [ ]:
spatial, spatial_stats = build_spatial_features(ds_peak)
print(f"Spatial features shape : {spatial.shape}")
print(f"Mean (should be ~0)    : {spatial.mean(0).abs().max():.4f}")
print(f"Std  (should be ~1)    : {spatial.std(0).mean():.4f}")

In [ ]:
feature_labels = [
    "sin_easting", "cos_easting",
    "sin_northing", "cos_northing",
    "sin_asp2k", "cos_asp2k",
    "sin_asp10k", "cos_asp10k",
    "station_height", "dem", "TPI",
    "slope_2k", "slope_10k",
    "SN_2k", "SN_10k", "WE_2k",
    "extra1", "extra2",
]

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(spatial.T.numpy(), aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set_yticks(range(spatial.shape[1]))
ax.set_yticklabels(feature_labels[:spatial.shape[1]], fontsize=7)
ax.set_xlabel("Station index")
ax.set_title("Normalised spatial features — all stations")
plt.colorbar(im, ax=ax, shrink=0.6, label="normalised value")
plt.tight_layout()
plt.show()

## 4 — Temporal encoding

In [ ]:
# Show how the 4 temporal features evolve over 48 hours
start = pd.Timestamp("2021-06-21 00:00:00+00:00")
steps = [start + pd.Timedelta(minutes=10 * i) for i in range(144)]  # 24h

temp_enc = torch.stack([encode_temporal(ts) for ts in steps])  # (144, 4)

labels = ["sin 24h", "cos 24h", "sin 365d", "cos 365d"]
fig, ax = plt.subplots(figsize=(12, 3))
for i, label in enumerate(labels):
    ax.plot(temp_enc[:, i].numpy(), label=label)
ax.set_xlabel("Timestep (10-min intervals)")
ax.set_title("Temporal encoding over 24 hours (21 June 2021)")
ax.legend(ncol=4)
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{int(x*10//60):02d}:{int(x*10%60):02d}")
)
plt.tight_layout()
plt.show()

## 5 — Dataset

In [ ]:
train_ds = StationMAEDataset(
    ds_peak,
    window_size=WINDOW_SIZE,
    delta_steps=DELTA_STEPS,
    split="train",
)
val_ds = StationMAEDataset(
    ds_peak,
    window_size=WINDOW_SIZE,
    delta_steps=DELTA_STEPS,
    split="val",
    obs_stats=train_ds.obs_stats,   # use training stats
)

print(f"Train samples : {len(train_ds):,}")
print(f"Val samples   : {len(val_ds):,}")

In [ ]:
# Inspect a single sample
sample = train_ds[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:15s}: {tuple(v.shape)}  {v.dtype}")
    else:
        print(f"  {k:15s}: {v}")

In [ ]:
# Plot one variable across the input window for a random sample
sample  = train_ds[500]
x       = sample["x"]       # (W, N, V)
x_mask  = sample["x_mask"]  # (W, N, V)
W, N, V = x.shape

VAR_IDX = 0   # temperature

fig, ax = plt.subplots(figsize=(12, 4))
for n in range(min(N, 30)):   # plot first 30 stations
    present = x_mask[:, n, VAR_IDX].bool()
    vals    = x[:, n, VAR_IDX].numpy()
    vals[~present.numpy()] = np.nan
    ax.plot(vals, alpha=0.4, linewidth=0.8)

ax.set_xlabel("Timestep (10-min)")
ax.set_ylabel("Normalised value")
ax.set_title(f"Input window — {VARIABLE_NAMES[VAR_IDX]} — first 30 stations")
plt.tight_layout()
plt.show()

In [ ]:
# Missing data heatmap across the input window
fig, axes = plt.subplots(1, V, figsize=(14, 4), sharey=True)
for v_idx, var in enumerate(VARIABLE_NAMES):
    ax  = axes[v_idx]
    mat = x_mask[:, :, v_idx].numpy()   # (W, N)
    ax.imshow(mat.T, aspect="auto", cmap="Greens", vmin=0, vmax=1, interpolation="none")
    ax.set_title(var, fontsize=8)
    ax.set_xlabel("Timestep")
    if v_idx == 0:
        ax.set_ylabel("Station")

fig.suptitle("Sensor availability — one sample window (green = present)", y=1.02)
plt.tight_layout()
plt.show()

## 6 — Embeddings

In [ ]:
spatial_emb_module  = SpatialEmbedding(d_model=D_MODEL)
temporal_emb_module = TemporalEmbedding(d_model=D_MODEL)
var_proj_module     = VariableProjection(d_model=D_MODEL)

# Use the first sample from the batch
x_in    = sample["x"].unsqueeze(0)       # (1, W, N, V)
m_in    = sample["x_mask"].unsqueeze(0)  # (1, W, N, V)
sp_in   = sample["spatial"].unsqueeze(0) # (1, N, 18)
te_in   = sample["x_temps"].unsqueeze(0) # (1, W, 4)

B, W, N, V_dim = x_in.shape

# Variable projection
x_flat  = x_in.view(B * W, N, V_dim)
m_flat  = m_in.view(B * W, N, V_dim)
vp_out  = var_proj_module(x_flat, m_flat).view(B, W, N, D_MODEL)

# Spatial embedding
sp_out  = spatial_emb_module(sp_in)     # (1, N, D_MODEL)

# Temporal embedding
te_out  = temporal_emb_module(te_in)    # (1, W, D_MODEL)

print(f"Variable projection output : {vp_out.shape}")
print(f"Spatial embedding output   : {sp_out.shape}")
print(f"Temporal embedding output  : {te_out.shape}")

In [ ]:
# Visualise spatial embedding vectors across stations
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(
    sp_out.squeeze(0).detach().numpy().T,  # (D_MODEL, N)
    aspect="auto", cmap="RdBu_r",
)
ax.set_xlabel("Station index")
ax.set_ylabel("d_model dimension")
ax.set_title("Spatial embedding (random init) — (N, d_model)")
plt.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Visualise temporal embedding across the input window
fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(
    te_out.squeeze(0).detach().numpy().T,  # (D_MODEL, W)
    aspect="auto", cmap="RdBu_r",
)
ax.set_xlabel("Window timestep")
ax.set_ylabel("d_model dimension")
ax.set_title("Temporal embedding — (W, d_model)")
plt.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

## 7 — Encoder

In [ ]:
encoder = StationMAEEncoder(
    d_model    = D_MODEL,
    num_heads  = 4,
    num_layers = 4,
    dropout    = 0.1,
    mask_ratio = 0.5,
)

total_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {total_params:,}")

In [ ]:
encoder.eval()
with torch.no_grad():
    encoded, masked_idx, visible_idx = encoder(
        x       = x_in,
        x_mask  = m_in,
        spatial = sp_in,
        x_temps = te_in,
    )

N_vis    = visible_idx.shape[1]
N_masked = masked_idx.shape[1]

print(f"Input shape           : {x_in.shape}       (B, W, N, V)")
print(f"Encoded shape         : {encoded.shape}  (B, W*N_vis, d_model)")
print(f"Visible stations      : {N_vis} / {N}")
print(f"Masked  stations      : {N_masked} / {N}")
print(f"Visible station idx   : {visible_idx[0].tolist()[:10]} ...")

In [ ]:
# Visualise encoded representation
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    encoded.squeeze(0).detach().numpy().T,   # (d_model, W*N_vis)
    aspect="auto", cmap="RdBu_r",
)
ax.set_xlabel(f"Token index (W={W} × N_vis={N_vis})")
ax.set_ylabel("d_model dimension")
ax.set_title("Encoder output — visible tokens only")
# Mark timestep boundaries
for w in range(1, W):
    ax.axvline(w * N_vis, color="white", linewidth=0.5, alpha=0.5)
plt.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# DataLoader test
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
batch  = next(iter(loader))

print("Batch shapes:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:15s}: {tuple(v.shape)}")

# Run full batch through encoder
encoder.eval()
with torch.no_grad():
    enc_out, m_idx, v_idx = encoder(
        x       = batch["x"],
        x_mask  = batch["x_mask"],
        spatial = batch["spatial"][0],   # (N, 18) — same for all samples
        x_temps = batch["x_temps"],
    )
print(f"\nEncoder output (full batch): {enc_out.shape}")